In [358]:
# Generic inputs for most ML tasks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
# This is new
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

pd.options.display.float_format = '{:,.2f}'.format

# setup interactive notebook mode
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from IPython.display import display, HTML

#### Read and pre-process data

In [359]:
# fetch data 

main_data = pd.read_csv('flight_data\mco_syr_sw_dep_20_21.csv')
main_data.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Destination Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),Wheels-off time,Taxi-Out time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,WN,1/1/2020,790.00,N499WN,BDL,8:10,8:06,160.00,144.00,-4.00,8:14,8.00,0.00,0.00,0.00,0.00,0.00
1,WN,1/1/2020,998.00,N928WN,IND,13:40,13:38,140.00,139.00,-2.00,13:49,11.00,0.00,0.00,0.00,0.00,0.00
2,WN,1/1/2020,"1,087.00",N709SW,PIT,21:50,21:49,135.00,135.00,-1.00,22:09,20.00,0.00,0.00,0.00,0.00,0.00
3,WN,1/1/2020,"1,227.00",N279WN,BNA,15:25,15:31,125.00,123.00,6.00,15:46,15.00,0.00,0.00,0.00,0.00,0.00
4,WN,1/1/2020,"1,336.00",N752SW,MKE,7:20,7:15,170.00,174.00,-5.00,7:29,14.00,0.00,0.00,0.00,0.00,0.00


In [360]:
main_data.dtypes

Carrier Code                                 object
Date (MM/DD/YYYY)                            object
Flight Number                               float64
Tail Number                                  object
Destination Airport                          object
Scheduled departure time                     object
Actual departure time                        object
Scheduled elapsed time (Minutes)            float64
Actual elapsed time (Minutes)               float64
Departure delay (Minutes)                   float64
Wheels-off time                              object
Taxi-Out time (Minutes)                     float64
Delay Carrier (Minutes)                     float64
Delay Weather (Minutes)                     float64
Delay National Aviation System (Minutes)    float64
Delay Security (Minutes)                    float64
Delay Late Aircraft Arrival (Minutes)       float64
dtype: object

In [361]:
len(main_data)

68850

In [362]:
main_data.rename(columns = {'Destination Airport' : 'Destination_Airport' , 
                            'Carrier Code' : 'Carrier_Code',
                            'Flight Number' : 'Flight_Number',
                            'Tail Number' : 'Tail_Number',
                            'Date (MM/DD/YYYY)':'Date', 
                            'Delay Carrier (Minutes)' : 'dep_Delay_Carrier',
                             'Delay Weather (Minutes)' : 'dep_Delay_Weather',
                 'Delay National Aviation System (Minutes)' : 'dep_Delay_National_Aviation_System',
                 'Delay Security (Minutes)' : 'dep_Delay_Security',
                 'Delay Late Aircraft Arrival (Minutes)' : 'dep_Delay_Late_Aircraft_Arrival'},inplace = True)

In [363]:
main_data.dtypes

Carrier_Code                           object
Date                                   object
Flight_Number                         float64
Tail_Number                            object
Destination_Airport                    object
Scheduled departure time               object
Actual departure time                  object
Scheduled elapsed time (Minutes)      float64
Actual elapsed time (Minutes)         float64
Departure delay (Minutes)             float64
Wheels-off time                        object
Taxi-Out time (Minutes)               float64
dep_Delay_Carrier                     float64
dep_Delay_Weather                     float64
dep_Delay_National_Aviation_System    float64
dep_Delay_Security                    float64
dep_Delay_Late_Aircraft_Arrival       float64
dtype: object

In [364]:
main_data = main_data[main_data['Destination_Airport'] == 'SYR']
len(main_data)

3

In [365]:
main_data['dep_hour'] = main_data['Scheduled departure time'].str.split(":").str[0].astype('int64')
main_data['Date'] = pd.to_datetime( main_data['Date'],format ="%m/%d/%Y") 
main_data['dep_day'] = main_data['Date'].dt.day_of_week.astype('int64') 
main_data['dep_year'] = main_data['Date'].dt.year.astype('int64')
main_data.dtypes

Carrier_Code                                  object
Date                                  datetime64[ns]
Flight_Number                                float64
Tail_Number                                   object
Destination_Airport                           object
Scheduled departure time                      object
Actual departure time                         object
Scheduled elapsed time (Minutes)             float64
Actual elapsed time (Minutes)                float64
Departure delay (Minutes)                    float64
Wheels-off time                               object
Taxi-Out time (Minutes)                      float64
dep_Delay_Carrier                            float64
dep_Delay_Weather                            float64
dep_Delay_National_Aviation_System           float64
dep_Delay_Security                           float64
dep_Delay_Late_Aircraft_Arrival              float64
dep_hour                                       int64
dep_day                                       

In [366]:
# main_data = main_data[main_data['dep_year'] == 2022]
# main_data.head()

In [367]:
main_data['dep_hour'].value_counts()

dep_hour
10    3
Name: count, dtype: int64

In [368]:
main_data['dep_year'].value_counts()

dep_year
2021    3
Name: count, dtype: int64

In [369]:
main_data['dep_order'] = 'latter'

In [370]:
main_data.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),...,Taxi-Out time (Minutes),dep_Delay_Carrier,dep_Delay_Weather,dep_Delay_National_Aviation_System,dep_Delay_Security,dep_Delay_Late_Aircraft_Arrival,dep_hour,dep_day,dep_year,dep_order
61054,WN,2021-11-20,"1,596.00",N938WN,SYR,10:30,10:21,170.00,156.00,-9.00,...,20.00,0.00,0.00,0.00,0.00,0.00,10,5,2021,latter
62534,WN,2021-11-27,"1,596.00",N212WN,SYR,10:30,10:28,170.00,160.00,-2.00,...,15.00,0.00,0.00,0.00,0.00,0.00,10,5,2021,latter
66122,WN,2021-12-18,"1,596.00",N557WN,SYR,10:30,10:47,170.00,143.00,17.00,...,12.00,0.00,0.00,0.00,0.00,0.00,10,5,2021,latter


In [371]:
# fetch data 

arr_data = pd.read_csv('flight_data\mco_syr_sw_arr_20_21.csv')
arr_data.head()

,Carrier Code,Date (MM/DD/YYYY),Flight Number,Tail Number,Origin Airport,Scheduled Arrival Time,Actual Arrival Time,Scheduled Elapsed Time (Minutes),Actual Elapsed Time (Minutes),Arrival Delay (Minutes),Wheels-on Time,Taxi-In time (Minutes),Delay Carrier (Minutes),Delay Weather (Minutes),Delay National Aviation System (Minutes),Delay Security (Minutes),Delay Late Aircraft Arrival (Minutes)
0,WN,11/14/2021,"1,634.00",N551WN,BWI,23:10,22:48,75.00,58.00,-22.00,22:46,2.00,0.00,0.00,0.00,0.00,0.00
1,WN,11/14/2021,"2,494.00",N266WN,BWI,11:15,10:54,75.00,65.00,-21.00,10:50,4.00,0.00,0.00,0.00,0.00,0.00
2,WN,11/14/2021,"2,496.00",N266WN,BWI,16:10,15:58,70.00,57.00,-12.00,15:55,3.00,0.00,0.00,0.00,0.00,0.00
3,WN,11/15/2021,"2,494.00",N420WN,BWI,11:15,0:00,75.00,0.00,0.00,0:00,0.00,0.00,0.00,0.00,0.00,0.00
4,WN,11/15/2021,"2,496.00",N773SA,BWI,16:10,16:04,70.00,70.00,-6.00,16:01,3.00,0.00,0.00,0.00,0.00,0.00


In [372]:
arr_data.rename(columns = {'Origin Airport' : 'Origin_Airport' , 
                            'Carrier Code' : 'Carrier_Code',
                            'Flight Number' : 'Flight_Number',
                            'Tail Number' : 'Tail_Number',
                            'Date (MM/DD/YYYY)':'Date', 
                            'Delay Carrier (Minutes)' : 'arr_Delay_Carrier',
                             'Delay Weather (Minutes)' : 'arr_Delay_Weather',
                 'Delay National Aviation System (Minutes)' : 'arr_Delay_National_Aviation_System',
                 'Delay Security (Minutes)' : 'arr_Delay_Security',
                 'Delay Late Aircraft Arrival (Minutes)' : 'arr_Delay_Late_Aircraft_Arrival'},inplace = True)

In [373]:
arr_data.dtypes

Carrier_Code                           object
Date                                   object
Flight_Number                         float64
Tail_Number                            object
Origin_Airport                         object
Scheduled Arrival Time                 object
Actual Arrival Time                    object
Scheduled Elapsed Time (Minutes)      float64
Actual Elapsed Time (Minutes)         float64
Arrival Delay (Minutes)               float64
Wheels-on Time                         object
Taxi-In time (Minutes)                float64
arr_Delay_Carrier                     float64
arr_Delay_Weather                     float64
arr_Delay_National_Aviation_System    float64
arr_Delay_Security                    float64
arr_Delay_Late_Aircraft_Arrival       float64
dtype: object

In [374]:
len(arr_data)

149

In [375]:
arr_data = arr_data[arr_data['Origin_Airport'] == 'MCO']
len(arr_data)

3

In [376]:
arr_data['arr_hour'] = arr_data['Scheduled Arrival Time'].str.split(":").str[0].astype('int64')

arr_data['Date'] = pd.to_datetime( arr_data['Date'],format ="%m/%d/%Y") 
arr_data['arr_day'] = arr_data['Date'].dt.day_of_week.astype('int64')
arr_data['arr_year'] = arr_data['Date'].dt.year.astype('int64')
arr_data.drop(columns = ['Scheduled Elapsed Time (Minutes)', 'Actual Elapsed Time (Minutes)'],inplace = True)

arr_data.dtypes

Carrier_Code                                  object
Date                                  datetime64[ns]
Flight_Number                                float64
Tail_Number                                   object
Origin_Airport                                object
Scheduled Arrival Time                        object
Actual Arrival Time                           object
Arrival Delay (Minutes)                      float64
Wheels-on Time                                object
Taxi-In time (Minutes)                       float64
arr_Delay_Carrier                            float64
arr_Delay_Weather                            float64
arr_Delay_National_Aviation_System           float64
arr_Delay_Security                           float64
arr_Delay_Late_Aircraft_Arrival              float64
arr_hour                                       int64
arr_day                                        int64
arr_year                                       int64
dtype: object

In [377]:
arr_data['arr_hour'].value_counts()

arr_hour
13    3
Name: count, dtype: int64

In [378]:
arr_data['arr_year'].value_counts()

arr_year
2021    3
Name: count, dtype: int64

In [379]:
merged_df = pd.merge(main_data, arr_data, on=['Carrier_Code','Date', 'Flight_Number', 'Tail_Number'], how='inner')
merged_df.head()

,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),Departure delay (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,WN,2021-11-20,"1,596.00",N938WN,SYR,10:30,10:21,170.00,156.00,-9.00,...,12:53,4.00,0.00,0.00,0.00,0.00,0.00,13,5,2021
1,WN,2021-11-27,"1,596.00",N212WN,SYR,10:30,10:28,170.00,160.00,-2.00,...,13:06,2.00,0.00,0.00,0.00,0.00,0.00,13,5,2021
2,WN,2021-12-18,"1,596.00",N557WN,SYR,10:30,10:47,170.00,143.00,17.00,...,13:07,3.00,0.00,0.00,0.00,0.00,0.00,13,5,2021


In [380]:
merged_df.dtypes


Carrier_Code                                  object
Date                                  datetime64[ns]
Flight_Number                                float64
Tail_Number                                   object
Destination_Airport                           object
Scheduled departure time                      object
Actual departure time                         object
Scheduled elapsed time (Minutes)             float64
Actual elapsed time (Minutes)                float64
Departure delay (Minutes)                    float64
Wheels-off time                               object
Taxi-Out time (Minutes)                      float64
dep_Delay_Carrier                            float64
dep_Delay_Weather                            float64
dep_Delay_National_Aviation_System           float64
dep_Delay_Security                           float64
dep_Delay_Late_Aircraft_Arrival              float64
dep_hour                                       int64
dep_day                                       

In [381]:
len(merged_df)

3

In [382]:
merged_df.to_csv('flight_data\mco_syr_sw_20_21.csv')

In [383]:
new_df = pd.read_csv('flight_data\mco_syr_sw_20_21.csv')
len(new_df)
new_df.head()

3

,Unnamed: 0,Carrier_Code,Date,Flight_Number,Tail_Number,Destination_Airport,Scheduled departure time,Actual departure time,Scheduled elapsed time (Minutes),Actual elapsed time (Minutes),...,Wheels-on Time,Taxi-In time (Minutes),arr_Delay_Carrier,arr_Delay_Weather,arr_Delay_National_Aviation_System,arr_Delay_Security,arr_Delay_Late_Aircraft_Arrival,arr_hour,arr_day,arr_year
0,0,WN,2021-11-20,"1,596.00",N938WN,SYR,10:30,10:21,170.00,156.00,...,12:53,4.00,0.00,0.00,0.00,0.00,0.00,13,5,2021
1,1,WN,2021-11-27,"1,596.00",N212WN,SYR,10:30,10:28,170.00,160.00,...,13:06,2.00,0.00,0.00,0.00,0.00,0.00,13,5,2021
2,2,WN,2021-12-18,"1,596.00",N557WN,SYR,10:30,10:47,170.00,143.00,...,13:07,3.00,0.00,0.00,0.00,0.00,0.00,13,5,2021
